
Con  riferimento  al  data  set  dell’esercitazione  su  clustering,  eseguire  una  classificazione 
binaria sulla feature death, una classificazione multiclasse sulla feature dzgroup ed una 
regressione sulla feature aps. 

Procedere  allo  split  train-test  secondo  il  rapporto  95%-15%  in  forma  stratificata 
secondo i valori della variabile target. 

2.Eseguire  l’imputazione  dei  dati  mancanti  con  le  stesse  strategie  dell’esercitazione 
precedente. 

3.Rimuovere  le  feature  che  presentano  elevata  correlazione  con  la  variabile  target  e 
successivamente analizzare le possibili feature multicollineari rimanenti. 

4.Utilizzare RandomForestClassifier per la classificazione e RandomForestRegressor per 
la regressione con i seguenti iperparametri(*) 
a. Classificatore 
i. criterion: “gini”, “log_loss” 
ii. min_samples_split: 2, 5, 10 
iii. max_features: “sqrt”, 5 
b. Regressore 
i. criterion: “squared_error”, “absolute_error” 
ii. min_samples_split: 2, 5, 10 
iii. max_features: “sqrt”, 5 

5.Valutare il regressore sul test set con la metrica R2, il classificatore binario con la curva 
ROC e la relativa AUC ed il classificatore multiclasse con le curve ROC e le AUC di ogni 
classe, ciascuna valutata in modalità one-vs-rest

In [1]:
from sklearn.model_selection import train_test_split
import pandas as pd 


df = pd.read_csv('dataset_esercitazione.csv')

df = df.dropna( subset=['aps']) 


X = df.drop(columns=['aps'])
y = df.aps 

X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.15, random_state=42)




In [2]:
from sklearn.impute import SimpleImputer

num_f = X_train.select_dtypes(include=['number']).columns.tolist()
cat_f = [ col for col in X_train.columns if col not in num_f ]

imputer1 = SimpleImputer(strategy='median').set_output(transform='pandas')
imputer2 = SimpleImputer(strategy='constant', fill_value='Unknown').set_output(transform='pandas')

X_tr_num = imputer1.fit_transform( X_train[num_f] )

X_te_num = imputer1.transform( X_test[num_f] )


X_tr_cat = imputer2.fit_transform( X_train[cat_f] )
X_te_cat = imputer2.transform( X_test[cat_f] )





In [3]:
from sklearn.preprocessing import OrdinalEncoder

encoder = OrdinalEncoder().set_output(transform='pandas')

X_tr_cat = encoder.fit_transform( X_tr_cat)

X_te_cat = encoder.transform( X_te_cat)

In [4]:
X_tr = pd.concat( [X_tr_num, X_tr_cat], axis=1)
X_te = pd.concat( [X_te_num, X_te_cat], axis=1)




In [5]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler().set_output(transform='pandas')

X_tr = scaler.fit_transform( X_tr )
X_te = scaler.transform( X_te )




In [6]:
corr_matrix = X_tr.corr().abs()

threshold = 0.7

num_col = len(corr_matrix.columns)


for i in range(num_col):
    for j in range(i+1,num_col):
        
        value = corr_matrix.iloc[i,j]
        
        if value > threshold:
            
            col = corr_matrix.columns[j]
            X_tr = X_tr.drop(columns=[col])
            X_te = X_te.drop(columns=[col])
            
            
            print(f'elimno colonna = {col}')
            


elimno colonna = totcst
elimno colonna = surv2m
elimno colonna = surv6m
elimno colonna = prg6m
elimno colonna = adlsc


In [7]:
from sklearn.feature_selection import SelectKBest, f_regression

sel = SelectKBest(score_func=f_regression, k=18)

print(y_train)
X_tr = sel.fit_transform( X_tr, y_train )

X_te = sel.transform( X_te )

print(X_tr.shape)

print(X_te.shape)

3717    70.0
3934    23.0
5401    37.0
7733    40.0
6604    54.0
        ... 
5735    25.0
5191    45.0
5390    28.0
860     12.0
7271    33.0
Name: aps, Length: 7738, dtype: float64
(7738, 18)
(1366, 18)


4.Utilizzare RandomForestClassifier per la classificazione e RandomForestRegressor per 
la regressione con i seguenti iperparametri(*) 
a. Classificatore 
i. criterion: “gini”, “log_loss” 
ii. min_samples_split: 2, 5, 10 
iii. max_features: “sqrt”, 5 
b. Regressore 
i. criterion: “squared_error”, “absolute_error” 
ii. min_samples_split: 2, 5, 10 
iii. max_features: “sqrt”, 5 

5.Valutare il regressore sul test set con la metrica R2, il classificatore binario con la curva 
ROC e la relativa AUC ed il classificatore multiclasse con le curve ROC e le AUC di ogni 
classe, ciascuna valutata in modalità one-vs-rest

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
from sklearn.model_selection import GridSearchCV

# 1. Griglia corretta con virgolette dritte
param_grid = {
    'criterion': ['squared_error', 'absolute_error'],
    'min_samples_split': [2, 5, 10],
    'max_features': ['sqrt', 5]
}

# 2. Configurazione corretta (scoring='r2', cv e n_jobs di sicurezza per Fedora)
grid_search = GridSearchCV(
    estimator=RandomForestRegressor(random_state=42),
    param_grid=param_grid,
    scoring='r2', 
)

# 3. Fit sul train set selezionato
grid_search.fit(X_tr, y_train)

# 4. Estrazione del modello migliore
best_regressor = grid_search.best_estimator_
best_param = grid_search.best_params_

# Nome del modello pronto per i titoli dei grafici
model_name = f"Regressor: crit={best_param['criterion']}, min_s={best_param['min_samples_split']}, max_f={best_param['max_features']}"
print(f"Migliori parametri trovati: {best_param}")


# A: Genera prima le predizioni numeriche usando il test set selezionato (X_te)
y_pred_test = best_regressor.predict(X_te)

# B: Calcola il punteggio cambiando il nome alla variabile finale per non romperla
r2_finale = r2_score(y_test, y_pred_test)

print(f"R2 Score finale sul Test Set: {r2_finale:.4f}")

Migliori parametri trovati: {'criterion': 'squared_error', 'max_features': 5, 'min_samples_split': 2}
R2 Score finale sul Test Set: 0.7461
